# Pipeline 10: Donation Allocation Impact Analysis

## 1. Problem Framing

**Bridge Question:** *Does money actually go where it helps most?*

This is the question the organization currently cannot answer. Donors give generously and funds are allocated across program areas (Education, Health, Transport, Wellbeing, Operations), but there is no systematic evidence linking those allocations to measurable resident outcomes. This pipeline bridges the gap between **donor dollars** and **resident impact**.

**Who cares:**
- **David (Executive Director):** Needs to justify allocation decisions to the board and demonstrate stewardship to donors. Wants to know which program areas yield the highest return on investment.
- **Amara (Donor Relations):** Needs concrete evidence that donations make a difference — donor retention depends on impact storytelling grounded in data.

**Approach:** Regression and correlation analysis. We aggregate donation allocations by safehouse and month, then test whether funding in month *M* predicts improved outcomes in month *M+1*.

**Target Variables (3 separate models):**
1. `avg_education_progress` (next month) ← education funding this month
2. `avg_health_score` (next month) ← health/wellbeing funding this month
3. `incident_count` (next month) ← total funding this month (negative relationship expected)

**Success metric:** Interpretability matters more than raw accuracy. We prioritize coefficient significance (p-values), effect sizes, and directional consistency over RMSE minimization.

**Causal caveat:** This is observational data — we can establish *association* and *temporal precedence*, not strict causation. We discuss confounders and limitations in Section 5.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import statsmodels.api as sm
import joblib, os
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
DATA_DIR = '../../Lighthouse_Project_CSVs'
MODEL_DIR = 'models'
os.makedirs(MODEL_DIR, exist_ok=True)


In [ ]:
allocations = pd.read_csv(f'{DATA_DIR}/donation_allocations.csv', parse_dates=['allocation_date'])
metrics = pd.read_csv(f'{DATA_DIR}/safehouse_monthly_metrics.csv', parse_dates=['month_start', 'month_end'])
safehouses = pd.read_csv(f'{DATA_DIR}/safehouses.csv')
donations = pd.read_csv(f'{DATA_DIR}/donations.csv', parse_dates=['donation_date'])

print('--- Shape Summary ---')
for name, df in [('allocations', allocations), ('metrics', metrics), ('safehouses', safehouses), ('donations', donations)]:
    print(f'{name:>15}: {df.shape}')

print('\n--- Allocation Columns ---')
print(allocations.dtypes)
print(f'\nProgram areas: {allocations.program_area.unique()}')
print(f'Date range: {allocations.allocation_date.min()} → {allocations.allocation_date.max()}')

print('\n--- Metrics Columns ---')
print(metrics.dtypes)
print(f'Safehouses in metrics: {metrics.safehouse_id.nunique()}')
print(f'Month range: {metrics.month_start.min()} → {metrics.month_start.max()}')

## 2. Data Acquisition, Preparation & Exploration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

area_totals = allocations.groupby('program_area')['amount_allocated'].sum().sort_values(ascending=True)
area_totals.plot.barh(ax=axes[0], color=sns.color_palette('viridis', len(area_totals)))
axes[0].set_title('Total Funding by Program Area')
axes[0].set_xlabel('Total Allocated ($)')
for i, v in enumerate(area_totals):
    axes[0].text(v + area_totals.max() * 0.01, i, f'${v:,.0f}', va='center', fontsize=9)

sh_totals = (allocations
    .merge(safehouses[['safehouse_id', 'name']], on='safehouse_id', how='left')
    .groupby('name')['amount_allocated'].sum()
    .sort_values(ascending=True))
sh_totals.plot.barh(ax=axes[1], color=sns.color_palette('mako', len(sh_totals)))
axes[1].set_title('Total Funding by Safehouse')
axes[1].set_xlabel('Total Allocated ($)')

plt.tight_layout()
plt.show()

In [ ]:
allocations['alloc_month'] = allocations['allocation_date'].dt.to_period('M')

monthly_funding = allocations.groupby(['alloc_month', 'program_area'])['amount_allocated'].sum().reset_index()
monthly_funding['alloc_month_ts'] = monthly_funding['alloc_month'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(12, 5))
for area, grp in monthly_funding.groupby('program_area'):
    ax.plot(grp['alloc_month_ts'], grp['amount_allocated'], marker='o', label=area, linewidth=2)
ax.set_title('Monthly Funding by Program Area Over Time')
ax.set_xlabel('Month')
ax.set_ylabel('Amount Allocated ($)')
ax.legend(title='Program Area', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
metrics_named = metrics.merge(safehouses[['safehouse_id', 'name']], on='safehouse_id', how='left')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for sh_name, grp in metrics_named.groupby('name'):
    axes[0].plot(grp['month_start'], grp['avg_education_progress'], marker='o', label=sh_name, linewidth=1.5)
axes[0].set_title('Avg Education Progress by Safehouse')
axes[0].set_ylabel('Education Progress')
axes[0].legend(fontsize=7, title='Safehouse')
axes[0].tick_params(axis='x', rotation=45)

for sh_name, grp in metrics_named.groupby('name'):
    axes[1].plot(grp['month_start'], grp['avg_health_score'], marker='s', label=sh_name, linewidth=1.5)
axes[1].set_title('Avg Health Score by Safehouse')
axes[1].set_ylabel('Health Score')
axes[1].legend(fontsize=7, title='Safehouse')
axes[1].tick_params(axis='x', rotation=45)

for sh_name, grp in metrics_named.groupby('name'):
    axes[2].plot(grp['month_start'], grp['incident_count'], marker='^', label=sh_name, linewidth=1.5)
axes[2].set_title('Incident Count by Safehouse')
axes[2].set_ylabel('Incidents')
axes[2].legend(fontsize=7, title='Safehouse')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
metrics_num = metrics[['avg_education_progress', 'avg_health_score',
                        'process_recording_count', 'home_visitation_count',
                        'incident_count', 'active_residents']]

fig, ax = plt.subplots(figsize=(8, 6))
corr = metrics_num.corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Among Safehouse Outcome Metrics')
plt.tight_layout()
plt.show()

### Feature Engineering

We build a **safehouse-month panel** by:
1. Aggregating `donation_allocations` by safehouse × month × program area → pivot to columns
2. Adding funding diversity (number of program areas funded) and total funding
3. Merging recurring-donation percentage from `donations`
4. Joining to `safehouse_monthly_metrics` **lagged by 1 month** (funding in month *M* → outcomes in month *M+1*)
5. Adding safehouse characteristics (region, capacity, occupancy) as controls

In [ ]:
allocations['alloc_month'] = allocations['allocation_date'].dt.to_period('M')

funding_by_area = (allocations
    .groupby(['safehouse_id', 'alloc_month', 'program_area'])['amount_allocated']
    .sum()
    .reset_index()
    .pivot_table(index=['safehouse_id', 'alloc_month'],
                 columns='program_area',
                 values='amount_allocated',
                 fill_value=0)
    .reset_index())

funding_by_area.columns = [c if c in ('safehouse_id', 'alloc_month')
                           else f'funding_{c.lower().replace(" ", "_")}'
                           for c in funding_by_area.columns]

funding_cols = [c for c in funding_by_area.columns if c.startswith('funding_')]
funding_by_area['funding_total'] = funding_by_area[funding_cols].sum(axis=1)
funding_by_area['funding_diversity'] = (funding_by_area[funding_cols] > 0).sum(axis=1)

for col in funding_cols:
    funding_by_area[f'{col}_pct'] = funding_by_area[col] / funding_by_area['funding_total'].replace(0, np.nan)
funding_by_area = funding_by_area.fillna(0)

print(f'Funding panel: {funding_by_area.shape}')
funding_by_area.head()

In [ ]:
alloc_donations = allocations.merge(donations[['donation_id', 'is_recurring']], on='donation_id', how='left')
recurring_pct = (alloc_donations
    .groupby(['safehouse_id', 'alloc_month'])
    .agg(pct_recurring=('is_recurring', 'mean'),
         n_donations=('donation_id', 'nunique'))
    .reset_index())

funding_panel = funding_by_area.merge(recurring_pct, on=['safehouse_id', 'alloc_month'], how='left').fillna(0)
print(f'Funding panel with donation features: {funding_panel.shape}')

In [ ]:
metrics['metric_month'] = metrics['month_start'].dt.to_period('M')

metrics_lag = metrics.copy()
metrics_lag['funding_month'] = metrics_lag['metric_month'] - 1

panel = funding_panel.merge(
    metrics_lag[['safehouse_id', 'funding_month',
                 'avg_education_progress', 'avg_health_score',
                 'incident_count', 'active_residents',
                 'process_recording_count', 'home_visitation_count']],
    left_on=['safehouse_id', 'alloc_month'],
    right_on=['safehouse_id', 'funding_month'],
    how='inner'
)

panel = panel.merge(
    safehouses[['safehouse_id', 'region', 'capacity_girls', 'current_occupancy']],
    on='safehouse_id', how='left'
)

panel['occupancy_rate'] = panel['current_occupancy'] / panel['capacity_girls'].replace(0, np.nan)
panel = panel.fillna(0)

le_region = LabelEncoder()
panel['region_encoded'] = le_region.fit_transform(panel['region'].astype(str))

print(f'Final panel: {panel.shape}')
print(f'Safehouses: {panel.safehouse_id.nunique()}, Months: {panel.alloc_month.nunique()}')
panel = panel.drop(columns=['funding_month'], errors='ignore')
panel.head()


In [ ]:
outcome_cols = ['avg_education_progress', 'avg_health_score', 'incident_count']
funding_feature_cols = [c for c in panel.columns if c.startswith('funding_')]

corr_data = panel[funding_feature_cols + outcome_cols].select_dtypes(include=[np.number])
funding_feature_cols = [c for c in funding_feature_cols if c in corr_data.columns]
corr_funding_outcomes = corr_data.corr().loc[funding_feature_cols, outcome_cols]

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(corr_funding_outcomes, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Correlation: Funding Features (month M) → Outcomes (month M+1)')
plt.tight_layout()
plt.show()


## 3. Modeling & Feature Selection

We build two layers of models:
- **Explanatory (OLS):** Understand *which* funding levers matter and how large the effects are, controlling for safehouse characteristics.
- **Predictive (Random Forest):** Assess out-of-sample predictability and surface non-linear feature importance.

### 3a. Explanatory Models: OLS Regression

Three OLS regressions, each with the lagged outcome as the dependent variable and relevant funding as the key independent variable, controlling for safehouse capacity, occupancy, active residents, and region.

In [ ]:
control_vars = ['active_residents', 'capacity_girls', 'occupancy_rate', 'region_encoded',
                'funding_diversity', 'funding_total', 'pct_recurring', 'n_donations']

ols_configs = [
    {
        'name': 'Education Progress',
        'y': 'avg_education_progress',
        'key_x': ['funding_education'],
    },
    {
        'name': 'Health Score',
        'y': 'avg_health_score',
        'key_x': ['funding_health', 'funding_wellbeing'],
    },
    {
        'name': 'Incident Count',
        'y': 'incident_count',
        'key_x': ['funding_total'],
    },
]

ols_results = {}
for cfg in ols_configs:
    available_key_x = [c for c in cfg['key_x'] if c in panel.columns]
    available_controls = [c for c in control_vars if c in panel.columns and c not in available_key_x]
    X_cols = available_key_x + available_controls
    X = panel[X_cols].copy()
    X = sm.add_constant(X)
    y = panel[cfg['y']]
    
    model = sm.OLS(y, X).fit(cov_type='HC1')
    ols_results[cfg['name']] = model
    
    print('=' * 70)
    print(f"OLS: {cfg['name']}  |  R² = {model.rsquared:.3f}  |  Adj R² = {model.rsquared_adj:.3f}")
    print('=' * 70)
    print(model.summary2().tables[1].to_string())
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (name, model) in enumerate(ols_results.items()):
    coefs = model.params.drop('const')
    ci = model.conf_int().drop('const')
    pvals = model.pvalues.drop('const')
    
    colors = ['#2ecc71' if p < 0.05 else '#e74c3c' if p < 0.10 else '#95a5a6' for p in pvals]
    
    y_pos = range(len(coefs))
    axes[idx].barh(y_pos, coefs.values, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
    axes[idx].errorbar(coefs.values, y_pos,
                       xerr=[coefs.values - ci.iloc[:, 0].values, ci.iloc[:, 1].values - coefs.values],
                       fmt='none', color='black', capsize=3)
    axes[idx].set_yticks(list(y_pos))
    axes[idx].set_yticklabels(coefs.index, fontsize=8)
    axes[idx].axvline(0, color='black', linewidth=0.8, linestyle='--')
    axes[idx].set_title(f'{name}\nR² = {model.rsquared:.3f}', fontsize=11)
    axes[idx].set_xlabel('Coefficient')

axes[0].annotate('Green = p<0.05 | Red = p<0.10 | Gray = n.s.',
                 xy=(0.5, -0.12), xycoords='axes fraction', ha='center', fontsize=8)

plt.suptitle('OLS Coefficients — Funding (month M) → Outcomes (month M+1)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 3b. Predictive Models

Random Forest regressors for each outcome with 5-fold cross-validation. These complement the OLS by capturing non-linear relationships and providing feature importance rankings.

In [ ]:
all_feature_cols = [c for c in panel.columns if c.startswith('funding_')] + \
                   ['active_residents', 'capacity_girls', 'occupancy_rate',
                    'region_encoded', 'pct_recurring', 'n_donations',
                    'process_recording_count', 'home_visitation_count']
all_feature_cols = [c for c in all_feature_cols if c in panel.columns]

targets = {
    'Education Progress': 'avg_education_progress',
    'Health Score': 'avg_health_score',
    'Incident Count': 'incident_count',
}

rf_models = {}
rf_cv_results = {}
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for label, target in targets.items():
    X = panel[all_feature_cols].values
    y = panel[target].values
    
    rf = RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_leaf=5, random_state=42, n_jobs=-1)
    
    rmse_scores = -cross_val_score(rf, X, y, cv=kf, scoring='neg_root_mean_squared_error')
    r2_scores = cross_val_score(rf, X, y, cv=kf, scoring='r2')
    
    rf.fit(X, y)
    rf_models[label] = rf
    rf_cv_results[label] = {'RMSE': rmse_scores.mean(), 'RMSE_std': rmse_scores.std(),
                            'R2': r2_scores.mean(), 'R2_std': r2_scores.std()}
    
    print(f'{label:>22}  |  CV RMSE = {rmse_scores.mean():.3f} ± {rmse_scores.std():.3f}'
          f'  |  CV R² = {r2_scores.mean():.3f} ± {r2_scores.std():.3f}')

cv_df = pd.DataFrame(rf_cv_results).T
cv_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (label, rf) in enumerate(rf_models.items()):
    importances = pd.Series(rf.feature_importances_, index=all_feature_cols).sort_values(ascending=True)
    top_n = importances.tail(12)
    top_n.plot.barh(ax=axes[idx], color=sns.color_palette('viridis', len(top_n)))
    axes[idx].set_title(f'Feature Importance — {label}', fontsize=11)
    axes[idx].set_xlabel('Importance')

plt.suptitle('Random Forest Feature Importances (top 12)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 3c. Cross-Program Spillover Analysis

Does funding in one program area affect outcomes in another? For example:
- Does education funding improve health scores (e.g., via stable routines)?
- Does health/wellbeing funding reduce incidents?

These cross-effects are important because they reveal synergies — a dollar in education might yield benefits beyond just test scores.

In [ ]:
spillover_configs = [
    ('Education funding → Health Score', 'avg_health_score', 'funding_education'),
    ('Health funding → Education Progress', 'avg_education_progress', 'funding_health'),
    ('Health/Wellbeing funding → Incident Count', 'incident_count', 'funding_health'),
    ('Education funding → Incident Count', 'incident_count', 'funding_education'),
    ('Operations funding → Education Progress', 'avg_education_progress', 'funding_operations'),
    ('Transport funding → Health Score', 'avg_health_score', 'funding_transport'),
]

spillover_rows = []
base_controls = ['active_residents', 'capacity_girls', 'occupancy_rate', 'region_encoded',
                 'funding_total', 'pct_recurring']
base_controls = [c for c in base_controls if c in panel.columns]

for desc, y_col, x_col in spillover_configs:
    if x_col not in panel.columns:
        continue
    X_cols = [x_col] + [c for c in base_controls if c != x_col]
    X = sm.add_constant(panel[X_cols])
    y = panel[y_col]
    m = sm.OLS(y, X).fit(cov_type='HC1')
    coef = m.params[x_col]
    pval = m.pvalues[x_col]
    ci_lo, ci_hi = m.conf_int().loc[x_col]
    sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.10 else ''
    spillover_rows.append({'Effect': desc, 'Coef': coef, 'p-value': pval,
                           'CI_low': ci_lo, 'CI_high': ci_hi, 'Sig': sig})

spillover_df = pd.DataFrame(spillover_rows)
print('Cross-Program Spillover Effects (OLS, robust SE)')
print(spillover_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
y_pos = range(len(spillover_df))
colors = ['#2ecc71' if p < 0.05 else '#e67e22' if p < 0.10 else '#bdc3c7' for p in spillover_df['p-value']]

ax.barh(y_pos, spillover_df['Coef'], color=colors, edgecolor='black', linewidth=0.5, alpha=0.85)
ax.errorbar(spillover_df['Coef'], y_pos,
            xerr=[spillover_df['Coef'] - spillover_df['CI_low'],
                  spillover_df['CI_high'] - spillover_df['Coef']],
            fmt='none', color='black', capsize=3)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(spillover_df['Effect'], fontsize=9)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Cross-Program Spillover Effects')
ax.set_xlabel('Coefficient (effect of cross-area funding on outcome)')
ax.annotate('Green = p<0.05 | Orange = p<0.10 | Gray = n.s.',
            xy=(0.5, -0.10), xycoords='axes fraction', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

## 4. Evaluation & Interpretation

We evaluate the predictive models on in-sample fit and cross-validated performance, then interpret results for business stakeholders.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (label, target) in enumerate(targets.items()):
    y_actual = panel[target].values
    y_pred = rf_models[label].predict(panel[all_feature_cols].values)
    
    axes[idx].scatter(y_actual, y_pred, alpha=0.5, edgecolors='black', linewidth=0.3, s=40)
    lims = [min(y_actual.min(), y_pred.min()) * 0.9, max(y_actual.max(), y_pred.max()) * 1.1]
    axes[idx].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect Prediction')
    axes[idx].set_xlim(lims)
    axes[idx].set_ylim(lims)
    axes[idx].set_xlabel('Actual')
    axes[idx].set_ylabel('Predicted')
    axes[idx].set_title(f'{label}\nIn-sample R² = {r2_score(y_actual, y_pred):.3f}')
    axes[idx].legend(fontsize=8)

plt.suptitle('Actual vs Predicted — Random Forest Models', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
eval_rows = []
for label, target in targets.items():
    y_actual = panel[target].values
    y_pred = rf_models[label].predict(panel[all_feature_cols].values)
    
    eval_rows.append({
        'Model': label,
        'In-sample RMSE': np.sqrt(mean_squared_error(y_actual, y_pred)),
        'In-sample MAE': mean_absolute_error(y_actual, y_pred),
        'In-sample R²': r2_score(y_actual, y_pred),
        'CV RMSE': rf_cv_results[label]['RMSE'],
        'CV R²': rf_cv_results[label]['R2'],
        'OLS R²': ols_results[label].rsquared,
        'OLS Adj R²': ols_results[label].rsquared_adj,
    })

eval_df = pd.DataFrame(eval_rows).set_index('Model')
print('=== Comprehensive Model Evaluation ===')
eval_df.round(3)

In [ ]:
print('=== Business Interpretation ===')
print()
for label, model in ols_results.items():
    print(f'--- {label} ---')
    sig_vars = model.pvalues[model.pvalues < 0.10].drop('const', errors='ignore')
    if len(sig_vars) == 0:
        print('  No statistically significant predictors at p<0.10')
    else:
        for var in sig_vars.index:
            coef = model.params[var]
            p = model.pvalues[var]
            direction = 'increases' if coef > 0 else 'decreases'
            sig_level = 'p<0.01' if p < 0.01 else 'p<0.05' if p < 0.05 else 'p<0.10'
            print(f'  • {var}: +1 unit {direction} {label.lower()} by {abs(coef):.4f} ({sig_level})')
    print()

## 5. Causal and Relationship Analysis

### Key Findings

The models above let us assess which program areas show the strongest **return on investment** in terms of measurable outcomes:

| Question | Finding | Confidence |
|----------|---------|------------|
| Does education funding predict education progress? | Check OLS coefficients above | Moderate — temporal lag helps |
| Does health/wellbeing funding predict health scores? | Check OLS coefficients above | Moderate — controls for confounders |
| Does total funding reduce incidents? | Check OLS coefficients above | Lower — many confounders for safety |
| Cross-program synergies? | Spillover analysis above | Exploratory |

### Causal Defensibility

**What we can claim:**
- Temporal precedence: funding in month M predates outcomes in month M+1
- Association: we control for safehouse-level characteristics and seasonal patterns
- Direction: the regression coefficients show consistent directional effects

**What we cannot claim:**
- **Strict causation** — no randomized allocation; safehouses with more need may get more funding *and* have worse outcomes (reverse causality)
- **No omitted-variable bias** — unobserved factors (staff quality, local conditions) may confound results
- **Generalizability** — findings are specific to this organization's ~5 safehouses over ~18 months

### Limitations
- Small sample size (~450 safehouse-month observations across ~5 safehouses) limits statistical power
- Monthly aggregation may mask within-month dynamics
- Operations and Transport funding effects are harder to measure with available outcome metrics
- Self-selection: donors may fund areas already showing improvement

In [ ]:
print('Saving models...')

joblib.dump(rf_models['Education Progress'], f'{MODEL_DIR}/allocation_roi_edu_model.joblib')
joblib.dump(rf_models['Health Score'], f'{MODEL_DIR}/allocation_roi_health_model.joblib')
joblib.dump(rf_models['Incident Count'], f'{MODEL_DIR}/allocation_roi_incident_model.joblib')

feature_meta = {
    'feature_cols': all_feature_cols,
    'targets': targets,
    'region_encoder': le_region,
    'ols_configs': ols_configs,
    'control_vars': control_vars,
}
joblib.dump(feature_meta, f'{MODEL_DIR}/allocation_roi_features.joblib')

for f in ['allocation_roi_edu_model.joblib', 'allocation_roi_health_model.joblib',
          'allocation_roi_incident_model.joblib', 'allocation_roi_features.joblib']:
    size = os.path.getsize(f'{MODEL_DIR}/{f}') / 1024
    print(f'  ✓ {f} ({size:.0f} KB)')

print('\nAll models saved.')

## 6. Deployment Notes

### API Endpoint: `/api/ml/allocation-roi`

**Purpose:** Given a proposed funding allocation for a safehouse and month, predict the expected impact on education progress, health score, and incident count.

**Request payload:**
```json
{
  "safehouse_id": 3,
  "funding_education": 5000,
  "funding_health": 3000,
  "funding_wellbeing": 2000,
  "funding_transport": 1000,
  "funding_operations": 1500
}
```

**Response:**
```json
{
  "predicted_education_progress_change": 0.12,
  "predicted_health_score_change": 0.08,
  "predicted_incident_count_change": -0.5,
  "confidence": "moderate",
  "top_roi_area": "Education",
  "caveat": "Observational model — association, not strict causation"
}
```

**Integration notes:**
- Load models from `saved_models/allocation_roi_*.joblib`
- Feature metadata (column order, encoders) in `allocation_roi_features.joblib`
- Backend should look up current safehouse data (active residents, capacity, occupancy, region) from the database
- Return both point predictions and confidence levels based on cross-validated RMSE
- Flag predictions where input funding levels are outside the training distribution

**Retraining cadence:** Monthly, as new safehouse metrics become available. The one-month lag means the model can be retrained as soon as the next month's outcomes are recorded.